In [1]:
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import time
import pennylane as qml
import dimod
import pandas as pd
from itertools import product
from scipy.optimize import minimize

import qubo_engine as qe

In [2]:
np.random.seed(42)

tickers = ["AAPL", "MSFT", "GOOGL",'JPM' , 'GS']
data = yf.download(tickers,start='2020-01-01', end='2026-01-01')
prices=data["Close"]
log_returns = np.log(prices/prices.shift(1)).dropna()

mean_returns= log_returns.mean().values *252
cov_matrix = log_returns.cov().values*252
sector_map={0:0,1:0,2:0,3:1,4:1}

n_assets=5
n_bits=2
N=n_assets*n_bits

# cov_matrix.head()
# mean_returns.head()


[*********************100%***********************]  5 of 5 completed


In [3]:
import sys
sys.path.append('/Users/linanachdi/Documents/GitHub/portfolio-optimization-engine')
import portfolio_engine as pe

start= time.time()

mw_weights,mw_ret,mw_vol,mw_sharpe = pe.maximum_sharpe(mean_returns,cov_matrix,sector_map=sector_map, sector_cap=0.6)

mw_time=time.time()-start

print("Constrained Markowitz weights:")
for ticker,w in zip(tickers,mw_weights):
    print(f"{ticker}:{w:.4f}")
print(f"Returns: {mw_ret:.2%} | Vol: {mw_vol:.2%} | Sharpe: {mw_sharpe:.4f}")
print(f"Time taken: {mw_time:.4f}s")

Constrained Markowitz weights:
AAPL:0.0000
MSFT:0.2903
GOOGL:0.3097
JPM:0.0415
GS:0.3585
Returns: 22.61% | Vol: 26.08% | Sharpe: 0.6868
Time taken: 0.0048s


In [4]:

Q= qe.build_qubo_matrix(mean_returns,cov_matrix,lambda_risk=1.0,lambda_budget=5.0,lambda_sector=2.0,sector_map=sector_map,
                        sector_cap=0.6,n_assets=n_assets,n_bits=n_bits)

start=time.time()
x_brute,energy_brute=qe.brute_force_qubo(Q)
time_brute=time.time() - start

start=time.time()
x_sa,energy_sa = qe.simulated_annealing_qubo(Q,num_reads=1000,num_sweeps=1000)
time_sa = time.time() - start

weights_brute = qe.decode_weights(x_brute,n_assets,n_bits)
weights_sa = qe.decode_weights(x_sa,n_assets,n_bits)

print(f"Brute force | energy: {energy_brute:.4f} | time: {time_brute:.4f}s")
print(f"Simulated Annealing | energy: {energy_sa:.4f} | time: {time_sa:.4f}s")
print("\nBrute force weights:")
for ticker,w_b in zip(tickers,weights_brute):
    print(f"{ticker}: {w_b:.4f}")
print("\nSA weights:")
for ticker,w_sa in zip(tickers,weights_sa):
    print(f"{ticker}:{w_sa:.4f}")

Brute force | energy: -6.4506 | time: 0.0023s
Simulated Annealing | energy: -6.4506 | time: 22.4460s

Brute force weights:
AAPL: 0.0000
MSFT: 0.3333
GOOGL: 0.3333
JPM: 0.0000
GS: 0.3333

SA weights:
AAPL:0.0000
MSFT:0.3333
GOOGL:0.3333
JPM:0.0000
GS:0.3333


In [5]:
# copy QAOA qubo to ising hamiltonian with cost and mixer layers from week 3

def qubo_to_ising_hamiltonian(Q):
    n=Q.shape[0]
    coeffs=[]
    obs=[]

    offset=0.0
    linear=np.zeros(n)
    quadratic={}

    for i in range(n):
        for j in range(n):
            if Q[i,j] == 0:
                continue
            if i==j:
                offset += Q[i,i]/2
                linear[i] += -Q[i,i]/2
            else:
                offset += Q[i,j]/4
                linear[i] += -Q[i,j]/4
                linear[j] += -Q[i,j]/4
                key=tuple(sorted((i,j)))
                quadratic[key]=quadratic.get(key,0) + Q[i,j]/4

    for i in range(n):
        if linear[i] != 0:
            coeffs.append(linear[i])
            obs.append(qml.PauliZ(i))
    for (i,j), coeff in quadratic.items():
        if coeff != 0:
            coeffs.append(coeff)
            obs.append(qml.PauliZ(i) @ qml.PauliZ(j))

    return coeffs, obs, offset

coeffs,obs,offset = qubo_to_ising_hamiltonian(Q)
cost_hamiltonian = qml.Hamiltonian(coeffs,obs)

n_qubits=N
dev=qml.device("default.qubit",wires=n_qubits)

def cost_layer(gamma,cost_hamiltonian):
    qml.templates.ApproxTimeEvolution(cost_hamiltonian,gamma,1)

def mixer_layer(beta):
    for i in range(n_qubits):
        qml.RX(2*beta,wires=i)

@qml.qnode(dev)
def qaoa_circuit(params,p):
    gammas = params[:p]
    betas = params[p:]

    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    for layer in range(p):
        cost_layer(gammas[layer],cost_hamiltonian)
        mixer_layer(betas[layer])

    return qml.expval(cost_hamiltonian)
    

In [6]:
def run_qaoa(p=1,n_restarts=5,maxiter=300):
    best_result,best_cost = None,np.inf
    history = []
    total_nfev = 0
    start_time=time.time()

    for restart in range(n_restarts):
        params0=np.random.uniform(0,np.pi,size=2*p)
        result= minimize(lambda params: qaoa_circuit(params,p), params0,method = "COBYLA", options={"maxiter":maxiter})
        history.append(result.fun)
        total_nfev += result.nfev
        if result.fun < best_cost:
            best_cost, best_result = result.fun , result

    time_taken = time.time() - start_time
    return best_result, best_cost,history,time_taken,total_nfev

result_p1, cost_p1,history_p1,time_p1,nfev_p1 = run_qaoa(p=1, n_restarts = 5)
result_p2, cost_p2, history_p2, time_p2, nfev_p2 = run_qaoa(p=2, n_restarts=15)

print(f"QAOA p=1 | energy: {cost_p1+offset:.4f} | time: {time_p1:.2f}s | evals: {nfev_p1}")
print(f"QAOA p=2 | energy: {cost_p2+offset:.4f} | time: {time_p2:.2f}s | evals: {nfev_p2}")

QAOA p=1 | energy: 11.0139 | time: 1.43s | evals: 190
QAOA p=2 | energy: 8.0070 | time: 40.28s | evals: 3631
